# PPT 발표 자료 출력(EDA, 학습 결과) 및 모델 학습 과정을 정리한 코드입니다.

# 0. 라이브러리 임포트 및 기본 설정
본 분석은 시군구별 데이터를 바탕으로 지방 소멸 위기 여부를 이진 분류하고, 이에 영향을 미치는 주요 요인을 머신러닝 앙상블 모델을 통해 도출하는 것을 목적으로 합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import os
import platform
import warnings
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

warnings.filterwarnings('ignore') # 경고 메시지 숨기기

# --- 한글 폰트 및 시각화 기본 설정 ---
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin': # Mac
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumGothic')

plt.rc('axes', unicode_minus=False)
sns.set_style("whitegrid")
os.makedirs('plots', exist_ok=True) # 시각화 저장용 폴더 생성

# --- 모델 성능 지표 시각화 및 자동 저장 함수 ---
def save_performance_plots(model, X_test, y_test, model_name):
    # 1. Confusion Matrix
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{model_name} Confusion Matrix')
    plt.savefig(f'plots/{model_name}_cm.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 2. ROC Curve
    y_score = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'AUC = {auc(fpr, tpr):.2f}', color='red', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'{model_name} ROC Curve')
    plt.legend(loc='lower right')
    plt.savefig(f'plots/{model_name}_roc.png', dpi=300, bbox_inches='tight')
    plt.show()

# 1. 한글 폰트 설정 (윈도우 환경)
plt.rcParams['font.family'] = 'Malgun Gothic'

# 2. 마이너스(-) 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 1. 데이터 불러오기 및 분석용 데이터프레임 생성

In [ ]:
import os
os.chdir('C://python//SKN25-2nd-2Team/')

In [ ]:
file_path = 'data/최종_전처리완료.csv' # 실제 환경에 맞게 경로 수정 필요
try:
    # thousands=',' 옵션으로 천 단위 콤마 제거
    df_full = pd.read_csv(file_path, encoding='cp949', thousands=',')
    print(f"'{file_path}' 성공적으로 로드 완료.")
except Exception as e:
    print(f"파일 로드 오류: {e}")
    df_full = pd.DataFrame()

if not df_full.empty:
    # 컬럼 인덱스를 통한 분석 변수 추출
    year_col_name = df_full.columns[5]       # 연도
    sigungu_col_name = df_full.columns[4]    # 시군구
    target_col_name = df_full.columns[11]    # 개선 인구 소멸 지수
    feature_cols = df_full.columns[12:81].tolist() # M열~CC열 Feature
    
    selected_columns = [year_col_name, sigungu_col_name, target_col_name] + feature_cols
    df = df_full[selected_columns].copy()
    
    # 데이터 형변환 (에러 발생 시 NaN)
    df[target_col_name] = pd.to_numeric(df[target_col_name], errors='coerce')
    df[year_col_name] = df[year_col_name].astype(int)
    
    print(f"분석 대상 Feature 개수: {len(feature_cols)}개")

# 2. 탐색적 데이터 분석 (EDA) 및 시각화 (PNG 저장)

In [ ]:
# --- 1. Target 변수 연도별 분포 (Boxplot) ---
plt.figure(figsize=(12, 6))

sns.boxplot(data=df, x=year_col_name, y=target_col_name, palette='husl')

plt.title(f'연도별 {target_col_name} 분포', fontsize=16)
plt.xlabel('연도', fontsize=12)
plt.ylabel('개선 인구 소멸 지수', fontsize=12)
plt.ylim(bottom=0)
plt.savefig('plots/1_target_boxplot_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- 2. 주요 변수 상관관계 히트맵 ---
corr_matrix = df.select_dtypes(include=[np.number]).corr(method='pearson', min_periods=1)
top_corr_features = corr_matrix[target_col_name].abs().sort_values(ascending=False).index[1:17]

plt.figure(figsize=(10, 8))
sns.heatmap(df[top_corr_features.insert(0, target_col_name)].corr(), 
            annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Target 및 주요 Feature 간의 상관관계 히트맵', fontsize=16, pad=15)
plt.savefig('plots/2_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- 3. 주요 Feature 시계열 트렌드 (Area Chart) ---
df_ts = df.copy()
df_grouped_mean = df_ts.groupby(year_col_name)[top_corr_features].mean().reset_index()
df_plot_melted = df_grouped_mean.melt(id_vars=year_col_name, var_name='Feature', value_name='Average Value')

year_min = df_ts[year_col_name].min()
year_max = df_ts[year_col_name].max()
unique_years = df_ts[year_col_name].unique()

def custom_area_plot(x, y, color, **kwargs):
    ax = plt.gca()
    ax.plot(x, y, linewidth=2.5, color=color, alpha=0.9)
    y_min, y_max = y.min(), y.max()
    y_margin = (y_max - y_min) * 0.2 if y_max != y_min else y_min * 0.2
    lower_bound = y_min - y_margin
    ax.set_ylim(lower_bound, y_max + y_margin)
    ax.fill_between(x, lower_bound, y, color=color, alpha=0.3)

g = sns.FacetGrid(df_plot_melted, col='Feature', hue='Feature', 
                  col_wrap=4, height=4, aspect=1.2, palette='Set2', 
                  sharey=False, sharex=False)
g.map(custom_area_plot, year_col_name, 'Average Value')

g.set(xticks=unique_years, xlim=(year_min, year_max))
g.set_titles(col_template="{col_name}", size=15, fontweight='bold')
g.set_axis_labels('연도', '평균값', fontsize=14)

for ax in g.axes.flatten():
    ax.tick_params(labelsize=12)

plt.subplots_adjust(top=0.88, hspace=0.3, wspace=0.3)
g.fig.suptitle('상관관계 상위 주요 Feature의 연도별 트렌드 (변수별 스케일 최적화)', fontsize=22, fontweight='bold')
plt.savefig('plots/3_timeseries_trend_area_chart.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# --- 4. 연도별 기초 통계량 및 Target 변수 분포 확인 ---
if not df.empty:
    print("--- 연도별 기초 통계량 ---")
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.float_format', '{:.2f}'.format):
        display(df.groupby(year_col_name).describe().T)

# 3. 소멸 위기 여부 생성 및 RF feature importance 기준 변수 선택

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

if not df.empty:
    years = sorted(df[year_col_name].unique())
    threshold = 1.04
    top_features_per_year = {}
    
    print(f"--- 연도별 RF 변수 중요도 기반 핵심 변수 추출 (Top 40) ---\n")
    
    for year in years:
        df_year = df[df[year_col_name] == year].copy()
        y_year = (df_year[target_col_name] < threshold).astype(int)
        
        X_year = df_year.drop(columns=[year_col_name, sigungu_col_name, target_col_name], errors='ignore')
        X_year = X_year.replace(',', '', regex=True).apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all')
        X_year = X_year.fillna(X_year.mean())
        
        rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(X_year, y_year)
        
        importances = pd.Series(rf.feature_importances_, index=X_year.columns)
        top_40 = importances.sort_values(ascending=False).head(40).index.tolist()
        
        top_features_per_year[year] = set(top_40)
        print(f"[{year}년] 변수 중요도 Top 40 추출 완료")

    common_features = set.intersection(*top_features_per_year.values())
    final_features = list(common_features)
    
    print(f"\n✅ 모든 연도 상위 40위 내 공통 핵심 변수 ({len(final_features)}개): \n{final_features}")

    if len(final_features) > 0:
        df_2024 = df[df[year_col_name] == 2024].copy()
        y = (df_2024[target_col_name] < threshold).astype(int)
        
        X_2024 = df_2024.drop(columns=[year_col_name, sigungu_col_name, target_col_name], errors='ignore')
        X_2024 = X_2024.replace(',', '', regex=True).apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all')
        X_2024 = X_2024.fillna(X_2024.mean())
        
        X_final_raw = X_2024[final_features]
        
        final_scaler = StandardScaler()
        X_final_scaled = pd.DataFrame(final_scaler.fit_transform(X_final_raw), 
                                      columns=X_final_raw.columns, index=X_final_raw.index)
        
        print("\n--- 최종 추출된 변수 스케일링 완료 (X_final_scaled) ---")
        display(X_final_scaled.head())
    else:
        print("\n❌ 모든 연도에 공통으로 속하는 Top 40 변수가 없습니다. Top N 개수를 늘려보세요.")